# Which invoice revision could this decision actually know?

One synthetic invoice, two revisions, three decision dates. Compare a deliberately
wrong event-date join with availability-aware financial / ML features, then edit
two timestamps yourself. **These invented CNY amounts demonstrate a data contract;
they do not establish model accuracy, investment returns or lending outcomes.**

一张合成发票，两个修订版本：一月份发生的数据，二月份和四月份的决策分别能看到什么？

Use Python 3.11+ and **Run all** in a fresh runtime. No clone, GPU, dataset account
or third-party runtime library is needed. Installation requires network access;
the example then runs locally in the notebook. Colab sign-in follows Google's
platform requirements.

## 1. Install this exact source snapshot

The full URL below selects **dev-belly/PITBridge**, pinned to commit
`a5e423debda20555a0cef4500606c9533908c6fd`. Use this source URL to identify this
project rather than relying on a package name alone. Pip may download build tools;
PITBridge itself has no third-party runtime dependencies.

In [ ]:
import subprocess
import sys

if sys.version_info < (3, 11):
    raise RuntimeError("This example requires Python 3.11 or newer")

# Exact source owner and commit, without Git or a source checkout.
SOURCE_COMMIT = "a5e423debda20555a0cef4500606c9533908c6fd"
SOURCE_URL = f"https://codeload.github.com/dev-belly/PITBridge/zip/{SOURCE_COMMIT}"
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet",
    "--disable-pip-version-check", "--force-reinstall", "--no-deps", SOURCE_URL,
])
print(f"Installed dev-belly/PITBridge from {SOURCE_COMMIT}")

## 2. Check the three original decisions

Both records describe January 31. The original amount is published February 5
and ingested February 6; the correction is published April 1 and ingested April 3.
Availability is **the later of publication and ingestion**, with a 90-day
freshness limit. The hand-calculated expectations are:

| Decision (UTC) | Expected result |
| --- | --- |
| February 1 | missing: `not_available` |
| February 15 | `invoice-v1`, CNY 100,000 |
| April 15 | `invoice-v2`, CNY 135,000 |

SQL must match both these fixed expectations and the independent Python
enumerator. These assertions belong to this unchanged fixture; the editable
scenario appears separately below.

In [ ]:
import sys
import pitbridge
from pitbridge import Decision, FeatureSpec, Observation, build_snapshot, reference_snapshot
from pitbridge.bundle import compare_unsafe

print(f"Python {sys.version.split()[0]}; PITBridge imported from {pitbridge.__file__}")

def invoice_observations(correction_ingested_at="2026-04-03T00:00:00Z"):
    return [
        Observation(
            record_id="invoice-v1", entity_id="SME-001", source="tax",
            feature="invoice_revenue", event_at="2026-01-31T00:00:00Z",
            published_at="2026-02-05T00:00:00Z", ingested_at="2026-02-06T00:00:00Z",
            revision=1, value=100000,
        ),
        Observation(
            record_id="invoice-v2", entity_id="SME-001", source="tax",
            feature="invoice_revenue", event_at="2026-01-31T00:00:00Z",
            published_at="2026-04-01T00:00:00Z", ingested_at=correction_ingested_at,
            revision=2, value=135000,
        ),
    ]

observations = invoice_observations()
decisions = [
    Decision("01-before-arrival", "SME-001", "2026-02-01T00:00:00Z"),
    Decision("02-original-known", "SME-001", "2026-02-15T00:00:00Z"),
    Decision("03-correction-known", "SME-001", "2026-04-15T00:00:00Z"),
]
specs = [FeatureSpec("tax", "invoice_revenue", max_age_days=90)]
snapshots = build_snapshot(observations, decisions, specs)
assert snapshots == reference_snapshot(observations, decisions, specs)
expected = [
    ("01-before-arrival", None, "not_available", None),
    ("02-original-known", 100000.0, "selected", "invoice-v1"),
    ("03-correction-known", 135000.0, "selected", "invoice-v2"),
]
actual = [(row.decision_id, row.value, row.status, row.record_id) for row in snapshots]
assert actual == expected, (actual, expected)
print("All three hand-calculated decisions match SQL and independent Python replay.")

## 3. See what the wrong join does

This is the same deliberately unsafe comparator used in the saved PITBridge
report. It filters only on event date and takes the highest revision, ignoring
availability, tombstones and freshness. **It is a counterexample, never a
feature-building recommendation.** Its April correction leaks into both February
decisions, even though all records share the same January event date.

In [ ]:
def show_comparison(source, rows):
    comparison = compare_unsafe(source, rows)
    print("Decision UTC | Wrong event-only record/value | Availability-aware result | Future knowledge?")
    for row, item in zip(rows, comparison, strict=True):
        wrong = f"{item['unsafe_record_id']}: {item['unsafe_value']}"
        safe = f"{row.status}, {row.record_id}: {row.value}"
        print(f"{row.decision_at[:10]} | {wrong} | {safe} | {item['unsafe_uses_future_knowledge']}")
    return comparison

comparison = show_comparison(observations, snapshots)
assert [row["unsafe_record_id"] for row in comparison] == ["invoice-v2"] * 3
assert [row["unsafe_uses_future_knowledge"] for row in comparison] == [True, True, False]
print("The wrong join uses future knowledge in 2 of these 3 invented decisions.")

## 4. Change two timestamps and rerun just the next cell

Try moving `decision_at` from April 2 to April 4, or move
`correction_ingested_at` from April 3 to April 1. Use timezone-aware ISO strings
(`Z` means UTC). The correction's publication date remains April 1: an earlier
ingestion timestamp cannot make it available before publication.

This separate scenario checks SQL against the independent enumerator. It does
not reuse the original fixture's fixed expectations, so valid edits can change
the result. The core contract will still reject invalid input timestamps.

In [ ]:
# Edit either string, then rerun this cell.
decision_at = "2026-04-02T00:00:00Z"
correction_ingested_at = "2026-04-03T00:00:00Z"

custom_observations = invoice_observations(correction_ingested_at)
custom_decisions = [Decision("editable-decision", "SME-001", decision_at)]
custom_rows = build_snapshot(custom_observations, custom_decisions, specs)
assert custom_rows == reference_snapshot(custom_observations, custom_decisions, specs)
print(f"Correction available at: {custom_observations[1].available_at}")
custom_comparison = show_comparison(custom_observations, custom_rows)

## Apply the timing question to your own feature pipeline

What timestamp in your feed establishes when a revision was genuinely available?
This notebook does not authenticate that upstream timestamp or demonstrate a
real trading or credit outcome. Read the [data contract](https://github.com/dev-belly/PITBridge/blob/main/docs/DATA_CONTRACT.md)
before adapting the API. For existing tables, the [optional CSV example](https://github.com/dev-belly/PITBridge/blob/main/examples/README.md#optional-pandascsv-input)
is a separate local adapter requiring pandas.

If a small synthetic counterexample exposes a mismatch with your workflow,
[share the timestamps, expected selection and observed result](https://github.com/dev-belly/PITBridge/issues/new?template=trial_feedback.md).